In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_csv  = (spark
           .read
           .format("csv")
           .option("header",True)
           .option("inferSchema",True)
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))
display(df_csv)

In [0]:
df_csv.filter(col("order_status") == 'Shipped').display()

In [0]:
df_json = (spark
           .read
           .format("JSON")
           .option("inferSchema",True)
           .option("mode","DROPMALFORMED")
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.json"))
display(df_json)

In [0]:
df_parquet = (spark
              .read
              .parquet("/Volumes/learnspark/raw/spark_volume/raw_orders/part-00000-tid-orders.c000.snappy.parquet"))
display(df_parquet)

In [0]:
df_csv.schema

In [0]:
schema = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_date', DateType(), True), StructField('product_id', StringType(), True), StructField('quantity', IntegerType(), True), StructField('price', DoubleType(), True), StructField('order_status', StringType(), True), StructField('shipping_address', StringType(), True), StructField('city', StringType(), True), StructField('country', StringType(), True), StructField('payment_method', StringType(), True), StructField('discount', DoubleType(), True), StructField('category', StringType(), True), StructField('sales_rep', StringType(), True), StructField('region', StringType(), True), StructField('ship_date', DateType(), True), StructField('delivery_days', IntegerType(), True), StructField('returned', StringType(), True), StructField('gender', StringType(), True)])

In [0]:
df_csv_1  = (spark
           .read
           .format("csv")
           .option("header",True)
           .option("inferSchema",True)
           .option("schema",schema)
           .load("/Volumes/learnspark/raw/spark_volume/raw_orders/orders.csv"))
display(df_csv_1.limit(1))

In [0]:
df_select = (df_csv
             .filter(col("category").isin("Books","Sports"))
             .select("order_id","customer_id",col("order_date"),df_csv.product_id,df_csv["quantity"],"order_status",col("shipping_address"),"city","country","payment_method","discount","category","sales_rep","region","ship_date","delivery_days","returned","gender")
)
display(df_select)

In [0]:
df_renamed = df_csv.withColumnRenamed("order_status","status")
display(df_renamed)


In [0]:
df_filename = df_renamed.withColumn("file_path",col("_metadata.file_path"))\
    .withColumn("file_name",col("_metadata.file_name"))
display(df_filename)

In [0]:
display(df_filename)

In [0]:
df = df_filename.withColumn("Total_Price",col("price")*col("quantity"))\
    .withColumn("Total_Price",round("Total_Price",2))\
        .sort(col("order_date").desc())
display(df)

In [0]:
df_cast = df_filename.withColumn("order_id",col("order_id").cast("STRING"))
display(df_cast)

In [0]:
df.sort(["order_date","quantity"],descending = [0,1]).display()

In [0]:
df_sort = df_filename.sort(col("product_id").asc())
display(df_sort)

In [0]:
df_sort = df_sort.sort(-5)
display(df_sort)

In [0]:
df_sort = df_sort.filter((df_sort.product_id == "P100") & (df_csv["status"].isin("Shipped","Cancelled"))).sort(["product_id","quantity"],ascending = [True,False]).limit(3)
display(df_sort)

In [0]:
display(df_filename)

In [0]:
df_drop = (
    df_filename
    .filter(col("order_date") > "2025-12-15")
    .sort(["product_id", "quantity"], ascending=[True, False])
    .withColumn("file", split(col("file_name"), r"\.")[0])
    .drop("file_name")
    .withColumnRenamed("file","file_name")
)
display(df_drop)

In [0]:
df_dedups = df_drop.dropDuplicates(subset=["order_date","product_id"])
display(df_dedups)

In [0]:
df_union = df_drop.union(df_dedups)
display(df_union)

In [0]:
df_change_col_order = df_drop.select("order_id","category",col("order_date"),"product_id","quantity","price","status","shipping_address","city","country","payment_method","discount","customer_id","sales_rep","region","ship_date","delivery_days","returned","gender",df_drop.file_path,df_drop["file_name"]
)

In [0]:
df_npunion = df_change_col_order.union(df_dedups)
display(df_npunion)

In [0]:
df_unionByname = df_change_col_order.unionByName(df_dedups)
display(df_unionByname)

In [0]:
df_curr = (df_filename.withColumn("file_name",split(col("file_name"), r"\.")[0])
           .withColumn("processed_date",current_timestamp())
           )
display(df_curr)

In [0]:
df_dateadd = df_curr.withColumn("processed_date",date_add("processed_date",7))
display(df_dateadd)

In [0]:
df_format = (
    df_curr.withColumn("processed_date",date_format("processed_date",'yyyy-MM-dd'))
    .withColumn("processed_date",date_add("processed_date",3))
    .withColumn("day_of_week",date_format("processed_date","EEEE"))
)
display(df_format)

In [0]:
%py
df_customers = spark.sql("""select * from gizmobox.bronze.v_customers""")
display(df_customers)

In [0]:
df_dropna = df_customers.dropna()
display(df_dropna)

In [0]:
df_dropnaall = df_customers.dropna('all')
display(df_dropnaall)

In [0]:
df_drop = df_customers.dropna(thresh= 1)
display(df_drop)

In [0]:
df_drop = df_customers.dropna(subset=["email","customer_id"],how = 'all')
display(df_drop)

In [0]:
df_fill = df_customers.na.fill(1802,"customer_id")
display(df_fill.filter(col("customer_id") == 1802))

In [0]:
df_fillemail = (
    df_customers.na.fill({"email": "Nagiligari@gmail.com","customer_id" : 1803})
)
display(df_fillemail.filter(
    (col("email") == "Nagiligari@gmail.com")|
    (col("customer_id") == 1803)
))



In [0]:
df_explode = (
    df_filename.withColumn("file_name",split("file_name",r"\."))
    .withColumn("file_explode",explode_outer("file_name"))
    .withColumn("flag",array_contains("file_name","orders"))
    )
display(df_explode)

In [0]:
display(
    df_format.withColumn("total_price",round(col("quantity")*col("price"),2))
    .groupBy("product_id")
    .agg(round(sum("total_price"),2).alias("sum_total")).
    sort(["product_id"],ascending = True)
)

In [0]:
display(
    df_format.withColumn("total_price",round(col("quantity")*col("price"),2))
    .groupBy("product_id","customer_id")
    .agg(
        round(sum("total_price"),2).alias("sum_total")
         ,round(avg("total_price"),2).alias("avg_price"))
    .sort("product_id",ascending = True)
)

In [0]:
display(
    df_csv.groupBy("product_id").agg(approx_count_distinct("customer_id").alias("distinct_customers")).sort("product_id",ascending = True)
)

In [0]:
display(
    df_csv.groupBy("product_id").agg(countDistinct("customer_id").alias("distinct_customers")).sort("product_id",ascending = True)
)

In [0]:
display(
    df_csv.groupBy("customer_id").agg(collect_list("product_id").alias("product_list")).sort("customer_id")
)

In [0]:
display(
    df_csv.groupBy("customer_id").agg(collect_set("product_id").alias("product_list")).sort("customer_id")
)

In [0]:
display(df_csv.groupBy("customer_id","product_id").pivot("order_status").agg(count("order_status")).sort("customer_id"))

In [0]:
df_pivot = (
    df_format.groupBy("customer_id").pivot("status").agg(count("status")).sort("customer_id")
    .withColumn(
        "Return_flag",
        when(col("Returned") == 1,"best_customer")
        .when(
        (col("Returned") <= 3),"good_customer"
        )
        .when(col("Returned") > 3,"bad_customer").otherwise("worst_customer")
        )
)

display(df_pivot)


In [0]:
data = [
    (1,),
    (0,),
    (1,),
    (None,)
]

columns = ["id"]

df1 = spark.createDataFrame(data,columns)
display(df1)

In [0]:
data = [
    (1,),
    (None,),
    (0,),
    (None,)
]

columns = ["id"]

df2 = spark.createDataFrame(data,columns)
display(df2)

In [0]:
df_inner = df1.join(df2,df1.id == df2.id,"inner")
display(df_inner)

In [0]:
df_left = df1.join(df2,df1.id == df2.id,"left")
display(df_left)

In [0]:
df_anti = df1.join(df2,df1.id == df2.id,"anti")
display(df_anti)

In [0]:
df_semi = df1.join(df2,df1.id == df2.id,"semi")
display(df_semi)

In [0]:
query = """WITH employee AS (
    SELECT 101 AS emp_id, 10 AS dept_id, 'Ajay' AS emp_name, 75000 AS salary
    UNION ALL
    SELECT 102, 10, 'Rahul', 68000
    UNION ALL
    SELECT 103, 20, 'Priya', 82000
    UNION ALL
    SELECT 104, 20, 'Sneha', 79000
    UNION ALL
    SELECT 105, 30, 'Vikram', 90000
    UNION ALL
    SELECT 106, 30, 'Kiran', 72000
    UNION ALL
    SELECT 107, 40, 'Meena', 65000
    UNION ALL
    SELECT 108, 40, 'Arun', 61000
    UNION ALL
    SELECT 109, 50, 'Divya', 88000
    UNION ALL
    SELECT 110, NULL, 'Ramesh', 55000
)
select * from employee"""

df_employee = spark.sql(query)
display(df_employee)

In [0]:
query = """WITH department AS (
    SELECT 10 AS dept_id, 'Engineering' AS dept_name, 'Suresh' AS hod
    UNION ALL
    SELECT 20, 'Finance', 'Lakshmi'
    UNION ALL
    SELECT 30, 'Human Resources', 'Anita'
    UNION ALL
    SELECT 40, 'Sales', 'Mahesh'
    UNION ALL
    SELECT 50, 'Analytics', 'Ravi'
    UNION ALL
    SELECT 60, 'Legal', 'Kavitha'
)
select * from  department"""

df_department = spark.sql(query)
display(df_department)

In [0]:
from pyspark.sql.window import Window
df_joined = df_employee.join(df_department,df_employee.dept_id == df_department.dept_id,"left").select("emp_id",df_employee.dept_id,col("emp_name"),df_employee["salary"],col("dept_name"),col("hod"))
df_joined = df_joined.na.drop("any")
df_window = (
    df_joined.withColumn("dense_rank",dense_rank().over(Window.partitionBy(col("dept_id")).orderBy("emp_id")))
    .withColumn("rank",rank().over(Window.partitionBy("dept_id").orderBy("emp_id")))
    .withColumn("row_number",row_number().over(Window.partitionBy("dept_id").orderBy("emp_id")))

             )
display(df_window)


In [0]:
data = [(2020,100),(2021,110),(2023,90),(2022,140),(2024,150)]
df = spark.createDataFrame(data,"Year INT,revenue INT")
display(df)

In [0]:
df_sum = df.withColumn("sum_total",sum("revenue").over(Window.orderBy("Year"))) # ByDefault it is considering unboundedPreceding and currentRow
display(df_sum)


In [0]:
df_sum = df.withColumn("sum_total",sum("revenue").over(Window.orderBy("Year").rowsBetween(Window.unboundedPreceding,Window.currentRow))) 
display(df_sum)

In [0]:
df_sum = df.withColumn("sum_total",sum("revenue").over(Window.orderBy("Year").rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))) 
display(df_sum)

In [0]:
df_sum = df.withColumn("sum_total",sum("revenue").over(Window.orderBy(col("Year").asc()).rowsBetween(Window.currentRow,Window.unboundedFollowing))) 
display(df_sum)

In [0]:
def square(x):
    return x*x
square(3.5)

In [0]:
square_udf = udf(square,FloatType())
df_sqre = df_curr.withColumn("price_square",square_udf(col("price")))
display(df_sqre)

In [0]:
@udf(returnType=IntegerType())
def my_square(x):
    return x*x
df_udf = df_curr.withColumn("sqaure_quanity",my_square("quantity"))
display(df_udf)

In [0]:
df_demo = spark.createDataFrame([("Hello World",),("just an Example",),("Hi Bro",)],["text"])
display(df_demo)

In [0]:
@udtf(returnType="word STRING")
class udtf_class:
    def eval(self,text:str): # Function name to be used
        for  word in text.split():
            yield (word,)

udtf_class(lit("Ajay is good boy")).show()


In [0]:
spark.udf.register("square",square_udf)
df_custom = df_curr.withColumn("total_square",call_udf("square",col("price")))
display(df_custom)

In [0]:
df_new = df_csv.filter(col("order_id").isin("1001","1002"))
df_new_1 = df_new.filter(col("order_id") == "1001").withColumn("product_id",lit("P1000"))
df_new_2 = df_new.filter(col("order_id") == "1002").withColumn("product_id",lit("P9990"))
df_new_3 = df_new.filter(col("order_id") == "1002").withColumn("order_id",lit("9001"))
df_new = df_new_1.union(df_new_2)
df_new = df_new.union(df_new_3)
display(df_new)

In [0]:
from delta.tables import DeltaTable
df_delta = DeltaTable.forPath(spark,"/Volumes/learnspark/raw/spark_volume/destination/delta/")

In [0]:
display(df_delta.alias("src")
 .merge(df_new.alias("tgt"),"src.order_id = tgt.order_id")
 .whenMatchedUpdateAll()
 .whenNotMatchedInsertAll()
 .execute())

In [0]:
df_delta = (spark.read.format("delta").load("/Volumes/learnspark/raw/spark_volume/destination/delta/"))
display(df_delta.filter(col("order_id").isin("1001","1002")))

In [0]:
df_csv.createOrReplaceTempView("orders_csv")

In [0]:
display(spark.sql("select * from orders_csv"))

In [0]:
from pyspark.sql import Row

data = [
    (1, "John", "US", 1200.50, "2026-01-15"),
    (2, "Alice", "UK", 850.75, "2026-02-10"),
    (3, "David", "FR", 1500.00, "2026-03-05"),
]

columns = [
    "customer_id",
    "customer_name",
    "country",
    "sales_amount",
    "order_date"
]

df1 = spark.createDataFrame(data, columns)

df1.createOrReplaceTempView("df1")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS learnspark.raw.orders_pratice
          (
              customer_id INT,
              customer_name STRING,
              country STRING,
              sales_amount DOUBLE,
              order_date DATE
          )
          """)

In [0]:
spark.sql("""
          INSERT INTO learnspark.raw.orders_pratice
          SELECT * FROM df1
          """)

In [0]:
%sql
select * from learnspark.raw.orders_pratice

In [0]:
from pyspark.sql import Row

data = [
    (4, "Ajay", "DE", 1590.00, "2026-12-15"),
    (2, "Alice", "UK", 960.75, "2026-02-10"),
    (5, "David", "IT", 1250.00, "2026-10-05"),
]

columns = [
    "customer_id",
    "customer_name",
    "country",
    "sales_amount",
    "order_date"
]

df2 = spark.createDataFrame(data, columns)

df1.createOrReplaceTempView("df2")

In [0]:
%py
(
    df2.alias("source").mergeInto("learnspark.raw.orders_pratice",col("source.customer_id") == col("learnspark.raw.orders_pratice.customer_id"))
    .whenMatched().update({"sales_amount":df2.sales_amount})
    .whenNotMatched().insertAll()
    .merge()
)


In [0]:
%sql
select * from learnspark.raw.orders_pratice

In [0]:
%sql
DESCRIBE DATABASE learnspark.raw

In [0]:
%sql
DESCRIBE TABLE learnspark.raw.orders_pratice

In [0]:
%sql
DESCRIBE QUERY SELECT * FROM learnspark.raw.orders_pratice

In [0]:
%sql
SHOW TABLES FROM learnspark.raw

In [0]:
%sql
USE learnspark.raw;
SHOW TABLE EXTENDED LIKE "orders_pratice";

In [0]:
%sql
SHOW TBLPROPERTIES learnspark.raw.orders_pratice

In [0]:
%sql
SHOW PARTITIONS learnspark.raw.orders_pratice

In [0]:
%sql
select * from orders_csv

In [0]:
%sql
SELECT customer_id, array_agg(order_id) FROM orders_csv group by customer_id

In [0]:
display(
    df_csv.groupBy("customer_id").agg(array_agg("order_id"))
)

In [0]:
%sql
SELECT customer_id,collect_list(order_id),collect_set(order_id) from  orders_csv group by customer_id

In [0]:
%sql
SELECT 
corr(price,quantity),
max(price),
min(price),
mean(price)
FROM orders_csv